In [1]:
import pandas as pd
import gspread #Conecta con la API de google sheets
import re #Para usar regex -> buscar palabras
import unicodedata #Normalizar texto (mayús y tildes)
from pathlib import Path


In [2]:

# DICCIONARIOS: grupos/keywords, reglas, puntajes
KEYWORDS= {
    "POLICIA":["prefecto", "bastones largos", "policia", "efectivo", "uniformado", "comisaria", "comisario", "policial", "GNA", "gendarmeria", "SPF", "penitenciario", "DIR", "direccion de despliegue de intervenciones rapidas", "SPB", "seguridad privada", "fuerzas de seguridad", "fuerzas federales"],
    "VIOLENCIA_POLICIAL":["megaoperativo", "desalojo", "gas", "gases", "lacrimogeno", "gatillo facil", "uso desmedido de la fuerza", "ejecucion reglamentaria", "tiro a matar", "fusilamiento", "violencia institucional", "violencia estatal", "reprimio", "represion", "reprimieron", "redujo", "redujeron", "fusilo", "fusilamiento", "aprehendio", "aprehendieron", "arresto", "gatillo", "bastones largos", "operativo", "antipiquete"],
    "CABA":["caba", "ciudad de buenos aires", "capital federal", "Almagro", "Balvanera", "Barracas", "Belgrano", "Boedo", "Caballito", "Chacarita", "Coghlan", "Colegiales", "Constitución", "Flores", "Floresta", "La Boca", "La Paternal", "Liniers", "Mataderos", "Monte Castro", "Monserrat", "Montevideo", "Nuñez", "Palermo", "Parque Avellaneda", "Parque Chacabuco", "Parque Chas", "Parque Patricios", "Puerto Madero", "Recoleta", "Retiro", "Saavedra", "San Cristóbal", "San Nicolás", "San Telmo", "Vélez Sarsfield", "Versalles", "Villa Crespo", "Villa del Parque", "Villa Devoto", "Villa General Mitre", "Villa Lugano", "Villa Luro", "Villa Ortúzar", "Villa Pueyrredón", "Villa Real", "Villa Riachuelo", "Villa Santa Rita", "Villa Soldati", "Villa Urquiza", "plaza de mayo", "congreso", "casa rosada", "legislatura porteña", "jefatura de gobierno porteño", "gobierno de la ciudad", "gobierno porteño", "ministerio de trabajo", "obelisco", "hospital argerich", "hospital fernández", "hospital pirovano", "hospital durand", "hospital ramos mejía", "hospital bonaparte", "hospital italiano", "microcentro", "centro porteño", "once", "tribunales", "retiro", "puerto madero", "costanera", "costanera sur", "costanera norte", "parque centenario", "parque lezama", "parque 3 de febrero", "bosques de palermo", "villa 31", "villa 21-24", "villa 1-11-14", "villa 15", "ciudad oculta", "barrio 31", "barrios populares", "barrio popular", "villa porteña", "villas porteñas"],
    "VIOLENCIA_GENERAL":["incidente", "disturbio", "golpe", "golpeo", "golpiza", "remato", "asesino", "abandono", "caceria", "violento", "abuso", "agredio", "baleo", "refriega", "bala", "molotov", "violencia", "quemarropa", "disparo", "disparado", "ejecuto", "mato"],
    "POSIBLE_VICTIMA":["docente", "universitario", "cientifico", "cientifica", "jubilado", "jubiladas", "protesta", "movilizacion", "gremial", "gremio", "militante", "grupo violento", "rostros ocultos", "anarquista", "trosko", "troskista", "rostros ocultos", "rostro oculto", "sindicalista", "piquetero", "mantero", "manifestante", "vendedor", "ambulante", "terrorista"],
    "VICTIMA":["arrestado", "arrestada", "demorado", "demorada", "detenido", "detenida", "desarmado", "desarmada", "aprehendido", "aprehendida", "golpeado", "golpeada", "victima", "situacion de calle", "resistencia a la autoridad", "indigente", "indigencia"]
}
# reglas ordenadas por prioridad
RULES = [
    {"name": "POLICIA + VICTIMA + VIOLENCIA POLICIAL + CABA", "groups": ["POLICIA", "VICTIMA", "VIOLENCIA_POLICIAL", "CABA"]},
    {"name": "POLICIA + POSIBLE VICTIMA + VIOLENCIA POLICIAL + CABA", "groups": ["POLICIA", "POSIBLE_VICTIMA", "VIOLENCIA_POLICIAL", "CABA"]},
    {"name": "POLICIA + VICTIMA + VIOLENCIA GENERAL + CABA", "groups": ["POLICIA", "VICTIMA", "VIOLENCIA_GENERAL", "CABA"]},
    {"name": "POLICIA + POSIBLE VICTIMA + VIOLENCIA GENERAL + CABA", "groups": ["POLICIA", "POSIBLE_VICTIMA", "VIOLENCIA_GENERAL", "CABA"]},
    {"name": "POLICIA + VICTIMA + CABA", "groups": ["POLICIA", "VICTIMA", "CABA"]},
    {"name": "POLICIA + POSIBLE VICTIMA + CABA", "groups": ["POLICIA", "POSIBLE_VICTIMA", "CABA"]},
    {"name": "POLICIA + VIOLENCIA POLICIAL + CABA", "groups": ["POLICIA", "VIOLENCIA_POLICIAL", "CABA"]},
    {"name": "POLICIA + VIOLENCIA GENERAL + CABA", "groups": ["POLICIA", "VIOLENCIA_GENERAL", "CABA"]},
    {"name": "POLICIA + VICTIMA + VIOLENCIA POLICIAL", "groups": ["POLICIA", "VICTIMA", "VIOLENCIA_POLICIAL"]},
    {"name": "POLICIA + VICTIMA + VIOLENCIA GENERAL", "groups": ["POLICIA", "VICTIMA", "VIOLENCIA_GENERAL"]},
    {"name": "POLICIA + POSIBLE VICTIMA + VIOLENCIA POLICIAL", "groups": ["POLICIA", "POSIBLE_VICTIMA", "VIOLENCIA_POLICIAL"]},
    {"name": "POLICIA + POSIBLE VICTIMA + VIOLENCIA GENERAL", "groups": ["POLICIA", "POSIBLE_VICTIMA", "VIOLENCIA_GENERAL"]},
    {"name": "POLICIA + VICTIMA", "groups": ["POLICIA", "VICTIMA"]},
    {"name": "POLICIA + POSIBLE VICTIMA", "groups": ["POLICIA", "POSIBLE_VICTIMA"]},
    {"name": "POLICIA + VIOLENCIA POLICIAL", "groups": ["POLICIA", "VIOLENCIA_POLICIAL"]},
    {"name": "POLICIA + VIOLENCIA GENERAL", "groups": ["POLICIA", "VIOLENCIA_GENERAL"]}
]

PUNTAJES_GRUPO = {
    "POLICIA": 3,
    "VIOLENCIA_POLICIAL": 3,
    "CABA": 4,
    "VICTIMA": 2,
    "POSIBLE_VICTIMA": 1,
    "VIOLENCIA_GENERAL": 1
}



In [3]:

# NORMALIZADOR: pasamos todo a minus y eliminamos tildes
# devuelve tipo str
def normalizar_texto(texto:str):
    texto = texto.lower()
    return "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )


In [4]:

# Rendimiento de texto en Python: No podés buscar palabra por palabra en un bucle simple; tardaría demasiado. La solución es precompilar expresiones regulares
# unificadas (una sola "regex" por categoría). Procesa miles de textos en segundos.
# Precompilamos una sola regex por categoría ordenada de frases largas a cortas.
COMPILADO = {}
for grupo, palabras in KEYWORDS.items():
    palabras_normalizadas = [normalizar_texto(p) for p in palabras]
    # prioriza frases compuestas / largas
    palabras_normalizadas.sort(key=len, reverse=True)
    # construye la expresión regular final como un string
    patron_unificado = rf"\b({'|'.join(re.escape(p) for p in palabras_normalizadas)})\b"
    # se compila para que luego sea más rapido buscar
    COMPILADO[grupo] = re.compile(patron_unificado)



In [ ]:

def analizar_articulo(texto:str):
    texto_normalizado = normalizar_texto(texto)

    grupos_detectados = {}
    palabras_detectadas = {}

    for grupo, regex in COMPILADO.items():
        coincidencias = list(set(regex.findall(texto_normalizado)))
        if coincidencias :
            grupos_detectados[grupo] = True
            palabras_detectadas[grupo] = coincidencias
        else:
            grupos_detectados[grupo] = False
    puntaje = sum(
        puntos for grupo, puntos in PUNTAJES_GRUPO.items()
    )


    regla_principal = "NINGUNA"
    for regla in RULES:
        if all(grupos_detectados.get(g,False) for g in regla["groups"]): #chequear groups
            regla_principal = regla["name"]
            break

    #resumen legible
    detalle_str = " | ".join(
        f"{g}: {', '.join(words)}" for g, words in palabras_detectadas.items()
    )

    return {
        "puntaje_total" : puntaje,
        "regla_principal": regla_principal,
        "policia": grupos_detectados.get("POLICIA", False),
        "violencia_policial": grupos_detectados.get("VIOLENCIA_POLICIAL", False),
        "violencia_general": grupos_detectados.get("VIOLENCIA_GENERAL", False),
        "caba": grupos_detectados.get("CABA", False),
        "victima": grupos_detectados.get("VICTIMA", False),
        "posible_victima": grupos_detectados.get("POSIBLE_VICTIMA", False),
        "palabras_detectadas": detalle_str
    }

In [6]:

def procesar_archivos_y_subir(
    carpeta_archivos: str, # la ruta de la carpeta donde están las noticias
    credenciales_json: str, # archivo de credenciales de Google Cloud
    spreadsheet_id_o_nombre: str,
    nombre_hoja: str = "Articulos"
):
    carpeta = Path(carpeta_archivos)
    archivos = list(carpeta.glob("*.txt"))

    if not archivos:
        print(f"No se encontraron archivos .txt en {carpeta_archivos}")
        return

    print(f"Iniciando procesamiento de {len(archivos)} archivos...")

    filas = []
    for i, archivo in enumerate(archivos, 1):
        try:
            texto = archivo.read_text(encoding="utf-8", errors="ignore")
            res = analizar_articulo(texto)
            res["archivo"] = archivo.name
            filas.append(res)
        except Exception as e:
            print(f"Error leyendo {archivo.name}: {e}") # si hay un error avanza con el siguiente

        # Cada 500 archivos procesados muestra un aviso en consola para que ver cómo avanza
        if i % 500 == 0:
            print(f"Procesados {i}/{len(archivos)}...")

    if not filas:
        print("No se pudo extraer información de ningún archivo.")
        return

    # Convertimos a DataFrame para ordenar fácilmente
    df = pd.DataFrame(filas)

    # Reordenamos columnas para que quede prolijo en Google Sheets
    columnas_orden = [
        "archivo", "puntaje", "regla_principal", "caba",
        "policia", "violencia_policial", "violencia_general",
        "victima", "posible_victima", "palabras_detectadas"
    ]
    df = df[columnas_orden]
    df = df.sort_values(by="puntaje", ascending=False) # las notas con mayor puntaje aparecen arriba

    print("Conectando con Google Sheets...")
    # Autenticación con Google Sheets usando Service Account
    gc = gspread.service_account(filename=credenciales_json)

    # Podés abrir por ID o por título del Sheet
    try:
        sh = gc.open_by_key(spreadsheet_id_o_nombre)
    except Exception:
        sh = gc.open(spreadsheet_id_o_nombre)

    ws = sh.worksheet(nombre_hoja)

    print("Subiendo datos en batch a Google Sheets...")
    # Convertimos las filas del DataFrame a lista de listas (sin los encabezados)
    filas_a_agregar = df.values.tolist()

    if filas_a_agregar:
        # Si la hoja está vacía, crea los encabezados
        if len(ws.get_all_values()) == 0:
            print("Hoja vacía detectada: creando fila de encabezados...")
            ws.append_row(columnas_orden, value_input_option="USER_ENTERED")

        print(f"Agregando {len(filas_a_agregar)} filas nuevas al final de la hoja...")
        ws.append_rows(filas_a_agregar, value_input_option="USER_ENTERED")
    else:
        print("No hay filas válidas para agregar.")

    print("¡Listo! Proceso finalizado correctamente.")


In [7]:

if __name__ == "__main__":
    # ajustar cuando tengamos lo real acá
    CARPETA_TXT = "./mis_noticias"
    CREDENCIALES = "credentials.json"
    SHEET_ID = "TU_SPREADSHEET_ID_ACA"  # Lo sacás de la URL de tu Google Sheet
    HOJA = "Hoja 1"
    procesar_archivos_y_subir(CARPETA_TXT, CREDENCIALES, SHEET_ID, HOJA)

Iniciando procesamiento de 5 archivos...
Error leyendo test399.txt: name 'g' is not defined
Error leyendo test398.txt: name 'g' is not defined
Error leyendo test400.txt: name 'g' is not defined
Error leyendo test401.txt: name 'g' is not defined
Error leyendo test402.txt: name 'g' is not defined
No se pudo extraer información de ningún archivo.
